<a href="https://colab.research.google.com/github/melanieesg/OT-Consulting-Exploracion/blob/main/Exploracion_OT_Consulting_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OT Consulting – Exploración de datos individual
**Etapa 2: Análisis y exploración de los datos**

## Objetivo
Revisar los archivos de OT Consulting para conocer su estructura, tipos de datos, información faltante y relaciones. El análisis se enfoca en la disponibilidad de colaboradores, sus habilidades y su asignación a proyectos. En esta etapa no se modifican los archivos originales.



## 1. Importación de archivos
Subir los ocho archivos Excel (seis catálogos, asignaciones de colaboradores y matriz de habilidades) y el respaldo JSON a Colab (no subir el PDF). Si se trabaja localmente, colocar los nueve archivos en la carpeta del notebook.

In [1]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display

archivos = ["Catalogo_Colaboradores.xlsx", "Catalogo_Clientes.xlsx",
            "Catalogo_Departamentos.xlsx", "Catalogo_Roles.xlsx",
            "Catalogo_Habilidades.xlsx", "Catalogo_Activos.xlsx",
            "Asignaciones_Colaboradores.xlsx", "Matriz_Habilidades.xlsx",
            "otc_backup_2026-11-17.json"]

if not all(Path(a).exists() for a in archivos):
    from google.colab import files
    files.upload()

excel = {}
for archivo in archivos[:-1]:
    # Se conservan los textos NA tal como vienen en el archivo.
    for hoja, df in pd.read_excel(archivo, sheet_name=None, keep_default_na=False).items():
        excel[archivo + " / " + hoja] = df

with open(archivos[-1], encoding="utf-8") as f:
    respaldo = json.load(f)

# No mostrar contraseñas ni registros personales en las evidencias.
json_tablas = {k: pd.DataFrame(v) for k, v in respaldo.items() if isinstance(v, list)}
print("Archivos Excel importados:", len(excel))
print("Colecciones JSON:", list(json_tablas))
print("Campos de control adicionales:", [k for k in respaldo if k not in json_tablas])

Saving Asignaciones_Colaboradores.xlsx to Asignaciones_Colaboradores.xlsx
Saving Catalogo_Activos.xlsx to Catalogo_Activos.xlsx
Saving Catalogo_Clientes.xlsx to Catalogo_Clientes.xlsx
Saving Catalogo_Colaboradores.xlsx to Catalogo_Colaboradores.xlsx
Saving Catalogo_Departamentos.xlsx to Catalogo_Departamentos.xlsx
Saving Catalogo_Habilidades.xlsx to Catalogo_Habilidades.xlsx
Saving Catalogo_Roles.xlsx to Catalogo_Roles.xlsx
Saving Matriz_Habilidades.xlsx to Matriz_Habilidades.xlsx
Saving otc_backup_2026-11-17.json to otc_backup_2026-11-17.json
Archivos Excel importados: 8
Colecciones JSON: ['colaboradores', 'usuarios', 'habilidades', 'roles', 'departamentos', 'matriz', 'clientes', 'asignaciones', 'activos']
Campos de control adicionales: ['secuenciaOTC']


## 2. Exploración archivo por archivo
Primero reviso la cantidad de filas, columnas y nombres de los campos. El JSON incluye listas y diccionarios dentro de algunas colecciones, por lo que se deben separar para analizarlos.

In [2]:
for fuente, df in {**excel, **json_tablas}.items():
    print(f"\n{fuente}: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(df.columns.tolist())


Catalogo_Colaboradores.xlsx / Colaboradores: 107 filas, 45 columnas
['ID', 'Nombre Completo', 'Género', 'Fecha Nacimiento', 'Teléfono', 'Correo Electrónico', 'Ciudad', 'Domicilio', 'RFC', 'CURP', 'NSS', 'Estatus', 'Fecha Incorporación', 'Fecha de Baja', 'Tipo Nómina', 'Departamento', 'Puesto', 'Sueldo Neto IMSS', 'Sueldo Asimilados/Sindicato', 'Sueldo Complementos', 'Sueldo Neto Total', 'Comisión Nómina', 'Impuestos IMSS', 'Sueldo Total Integrado', 'SGMM', 'Seguro Vida', 'Beneficiario', 'Parentesco', 'Porcentaje Beneficiario', 'Alergias/Condiciones', 'Notas', 'Doc: Acta', 'Doc: INE', 'Doc: CURP', 'Doc: Pasaporte', 'Doc: Visa', 'Doc: Const. Situación Fiscal', 'Doc: NSS', 'Doc: Comprobante Domicilio', 'Doc: Banco', 'Doc: Foto', 'Doc: Contrato', 'Aviso: Infonavit', 'Aviso: Manutención', 'Formato: RH']

Catalogo_Clientes.xlsx / Clientes_Proyectos: 12 filas, 13 columnas
['Cód Cliente', 'Cliente', 'Industria', 'RFC', 'Contacto Principal', 'Email Contacto', 'Estatus Cliente', 'Proyecto', 'Fe

**Observación:** Se importaron 8 archivos Excel y 9 colecciones del JSON. Los catálogos tienen las mismas filas en ambas fuentes: 107 colaboradores, 55 roles, 71 habilidades, 6 departamentos y 123 activos. El Excel de clientes tiene 12 filas porque cada fila es un proyecto; el JSON agrupa esos proyectos en 8 clientes. El Excel de asignaciones tiene 31 filas, igual que las asignaciones separadas del JSON, y la matriz tiene 54 colaboradores en Excel (una columna por habilidad) y 56 registros en el JSON.

## 3. Metadatos y perfil de cada columna
Reviso el tipo original, los tipos de valores presentes, la cantidad de datos ausentes o vacíos y los valores distintos. No muestro los valores personales ni credenciales.

In [3]:
def faltante(s):
    return s.isna() | s.map(lambda x: isinstance(x, str) and not x.strip())

def perfil(df):
    filas = []
    for campo in df.columns:
        s = df[campo]
        vacios = faltante(s)
        presentes = s[~vacios]
        filas.append({"campo": campo, "tipo_pandas": str(s.dtype),
                      "tipos_observados": ", ".join(sorted({type(x).__name__ for x in presentes})),
                      "ausentes": int(s.isna().sum()),
                      "textos_vacios": int((vacios & ~s.isna()).sum()),
                      "% faltantes": round(100*vacios.mean(), 2) if len(s) else 0,
                      "distintos": presentes.map(repr).nunique()})
    return pd.DataFrame(filas)

for fuente, df in {**excel, **json_tablas}.items():
    print("\n", fuente)
    display(perfil(df))


 Catalogo_Colaboradores.xlsx / Colaboradores


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,ID,object,str,0,0,0.00,107
1,Nombre Completo,object,str,0,0,0.00,107
2,Género,object,str,0,0,0.00,2
3,Fecha Nacimiento,object,str,0,1,0.93,104
4,Teléfono,int64,int,0,0,0.00,1
5,Correo Electrónico,object,str,0,0,0.00,1
6,Ciudad,object,str,0,0,0.00,25
7,Domicilio,object,str,0,0,0.00,1
8,RFC,object,str,0,0,0.00,1
9,CURP,object,str,0,0,0.00,1



 Catalogo_Clientes.xlsx / Clientes_Proyectos


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,Cód Cliente,object,str,0,0,0.00,8
1,Cliente,object,str,0,0,0.00,8
2,Industria,object,str,0,0,0.00,5
3,RFC,object,str,0,8,66.67,4
4,Contacto Principal,object,str,0,0,0.00,8
5,Email Contacto,object,str,0,3,25.00,7
6,Estatus Cliente,object,str,0,0,0.00,1
7,Proyecto,object,str,0,0,0.00,12
8,Fecha Inicio,object,str,0,6,50.00,6
9,Fecha Fin,object,str,0,9,75.00,3



 Catalogo_Departamentos.xlsx / Departamentos


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,responsable,object,,0,6,100.0,0
1,objetivo,object,str,0,0,0.0,6
2,estatus,object,str,0,0,0.0,1
3,codigo,object,str,0,0,0.0,6
4,centroCostos,object,,0,6,100.0,0
5,uid,int64,int,0,0,0.0,6
6,nombre,object,str,0,0,0.0,6


... (24 outputs left out)


 asignaciones


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,proyectos,object,list,0,0,0.0,30
1,uid_colaborador,object,str,0,0,0.0,30



 activos


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,asignaciones,object,list,0,0,0.00,92
1,proveedor,object,str,0,70,56.91,6
2,tipo,object,str,0,0,0.00,2
3,fgarantia,object,,0,123,100.00,0
4,fcompra,object,str,0,70,56.91,35
5,desc,object,str,0,0,0.00,2
6,uom,object,str,0,0,0.00,1
7,notas,object,str,0,115,93.50,5
8,serie,object,str,0,0,0.00,123
9,codigo,object,str,0,0,0.00,123


**Observación:** Roles y habilidades están completos. En departamentos, el responsable y el centro de costos están vacíos al 100 %. En clientes faltan todos los teléfonos y direcciones, y la mitad no tiene RFC ni ubicación. En activos, la fecha de garantía está vacía al 100 % y el proveedor y la fecha de compra faltan en el 56.91 %. En la matriz de Excel, el departamento está vacío en el 94.44 % de los colaboradores. Los archivos de colaboradores y usuarios contienen datos personales y contraseñas, por eso solo se muestran conteos.

## 4. Separación de datos anidados del JSON
Cada fila de proyectos representa un proyecto de un cliente; cada fila de asignaciones representa la participación de un colaborador en un proyecto. La matriz se convierte en una fila por combinación de colaborador y habilidad.

In [4]:
c = json_tablas["colaboradores"]
p = pd.DataFrame([dict(proyecto, uid_cliente=r["uid"], codigo_cliente=r["codigo"])
                  for r in respaldo["clientes"] for proyecto in r.get("proyectos", [])])
a = pd.DataFrame([dict(proyecto, uid_colaborador=r["uid_colaborador"])
                  for r in respaldo["asignaciones"] for proyecto in r.get("proyectos", [])])
h = pd.DataFrame([{"uid_colaborador": r["uid_colaborador"], "codigo_habilidad": k, "nivel": v}
                  for r in respaldo["matriz"] for k, v in r.get("habilidades", {}).items()])
act = pd.DataFrame([dict(asig, codigo_activo=r["codigo"])
                    for r in respaldo["activos"] for asig in r.get("asignaciones", [])])
for nombre, df in {"Proyectos":p, "Asignaciones":a, "Detalle habilidades":h, "Asignaciones activos":act}.items():
    print(nombre, df.shape)
    display(perfil(df))

Proyectos (12, 10)


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,email_cli,object,str,0,11,91.67,1
1,nombre,object,str,0,0,0.00,12
2,tel_cli,object,str,0,11,91.67,1
3,resp_cli,object,str,0,6,50.00,6
4,inicio,object,str,0,6,50.00,6
5,fin,object,str,0,9,75.00,3
6,estatus,object,str,0,0,0.00,2
7,pm_otc,object,str,0,6,50.00,3
8,uid_cliente,object,str,0,0,0.00,8
9,codigo_cliente,object,str,0,0,0.00,8


Asignaciones (31, 7)


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,inicio,object,str,0,0,0.00,4
1,tipo,object,str,0,0,0.00,2
2,rol,object,str,0,0,0.00,2
3,fin,object,str,0,24,77.42,3
4,estatus,object,str,0,0,0.00,1
5,proyecto_val,object,str,0,0,0.00,4
6,uid_colaborador,object,str,0,0,0.00,30


Detalle habilidades (1539, 3)


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,uid_colaborador,object,str,0,0,0.0,56
1,codigo_habilidad,object,str,0,0,0.0,71
2,nivel,object,str,0,0,0.0,4


Asignaciones activos (91, 5)


,campo,tipo_pandas,tipos_observados,ausentes,textos_vacios,% faltantes,distintos
0,notas,object,str,0,90,98.90,1
1,f_fin,object,str,0,81,89.01,9
2,uid_colab,object,str,0,0,0.00,73
3,f_ini,object,str,0,0,0.00,48
4,codigo_activo,object,str,0,0,0.00,91


**Observación:** El respaldo tiene 30 colaboradores con asignaciones, que suman 31 asignaciones. La matriz tiene 56 registros y 1,539 combinaciones de colaborador y habilidad, así que no deben confundirse estos dos niveles de detalle. Además, las 31 asignaciones se concentran en solo 4 de los 12 proyectos.

## 5. Datos faltantes e inconsistencias
Reviso los campos que tienen impacto directo sobre las preguntas de análisis. Las fechas de fin vacías pueden representar asignaciones indefinidas, no necesariamente errores.

In [5]:
activos = c["estatus"].eq("Activo")
bajas = c["estatus"].eq("Baja")
matriz = json_tablas["matriz"]
controles = [
    ("Colaboradores sin departamento", faltante(c["depto"]).sum(), len(c)),
    ("Puestos pendientes", c["puesto"].eq("PENDIENTE").sum(), len(c)),
    ("Asignaciones sin fecha fin", faltante(a["fin"]).sum(), len(a)),
    ("Proyectos sin fecha fin", faltante(p["fin"]).sum(), len(p)),
    ("Colaboradores activos sin registro en matriz", (activos & ~c["uid"].isin(matriz["uid_colaborador"])).sum(), activos.sum()),
    ("Registros de matriz sin colaborador", (~matriz["uid_colaborador"].isin(c["uid"])).sum(), len(matriz)),
    ("Colaboradores activos con sueldo integrado cero", (activos & c["sueldo_total_integrado"].eq(0)).sum(), activos.sum()),
    ("Bajas sin fecha de salida", (bajas & faltante(c["fechabaja"])).sum(), bajas.sum())
]
reporte = pd.DataFrame(controles, columns=["Hallazgo", "Afectados", "Total de referencia"])
reporte["Porcentaje"] = (100*reporte["Afectados"]/reporte["Total de referencia"]).round(2)
display(reporte)

,Hallazgo,Afectados,Total de referencia,Porcentaje
0,Colaboradores sin departamento,95,107,88.79
1,Puestos pendientes,48,107,44.86
2,Asignaciones sin fecha fin,24,31,77.42
3,Proyectos sin fecha fin,9,12,75.00
4,Colaboradores activos sin registro en matriz,28,75,37.33
5,Registros de matriz sin colaborador,2,56,3.57
6,Colaboradores activos con sueldo integrado cero,5,75,6.67
7,Bajas sin fecha de salida,18,32,56.25


**Observación:** El 88.79 % de los colaboradores no tiene departamento y el 44.86 % tiene el puesto como "PENDIENTE", lo que dificulta clasificarlos. 18 de las 32 bajas (56.25 %) no tienen fecha de salida. 28 de los 75 colaboradores activos (37.33 %) no aparecen en la matriz de habilidades y 5 tienen sueldo integrado en cero, lo que debe revisarse antes de calcular costos. El 77.42 % de las asignaciones no tiene fecha de fin; esto no es necesariamente un error, porque pueden ser asignaciones indefinidas, pero hay que confirmar la regla del negocio antes de medir disponibilidad.

## 6. Duplicados e integridad de las relaciones
Valido las llaves de los catálogos y compruebo que los registros relacionados tengan una referencia válida.

In [6]:
llaves = {"colaboradores":"id", "clientes":"codigo", "departamentos":"codigo",
          "roles":"codigo", "habilidades":"codigo", "activos":"codigo", "matriz":"uid_colaborador"}
for tabla, llave in llaves.items():
    s = json_tablas[tabla][llave]
    print(tabla, "llaves faltantes:", faltante(s).sum(), "duplicados:", s[~faltante(s)].duplicated().sum())
print("Combinaciones colaborador-habilidad duplicadas:", h.duplicated(["uid_colaborador", "codigo_habilidad"]).sum())
print("Asignaciones sin colaborador:", (~a.uid_colaborador.isin(c.uid)).sum())
print("Registros de matriz sin colaborador:", (~json_tablas["matriz"].uid_colaborador.isin(c.uid)).sum())
print("Habilidades sin código en catálogo:", (~h.codigo_habilidad.isin(json_tablas["habilidades"].codigo)).sum())
# El identificador original de proyecto combina el UID de cliente y el nombre del proyecto.
proyecto_clave = p.uid_cliente.astype(str) + "|" + p.nombre.astype(str)
print("Asignaciones sin proyecto correspondiente:", (~a.proyecto_val.isin(proyecto_clave)).sum())
print("Roles de asignaciones sin coincidencia:", (~a.rol.isin(json_tablas["roles"].nombre)).sum())
print("Activos asignados sin colaborador:", (~act.uid_colab.isin(c.uid)).sum())

colaboradores llaves faltantes: 0 duplicados: 0
clientes llaves faltantes: 0 duplicados: 0
departamentos llaves faltantes: 0 duplicados: 0
roles llaves faltantes: 0 duplicados: 0
habilidades llaves faltantes: 0 duplicados: 0
activos llaves faltantes: 0 duplicados: 0
matriz llaves faltantes: 0 duplicados: 0
Combinaciones colaborador-habilidad duplicadas: 0
Asignaciones sin colaborador: 0
Registros de matriz sin colaborador: 2
Habilidades sin código en catálogo: 0
Asignaciones sin proyecto correspondiente: 0
Roles de asignaciones sin coincidencia: 0
Activos asignados sin colaborador: 0


**Observación:** No hay llaves faltantes ni duplicadas, y las asignaciones, habilidades, roles y activos asignados tienen una referencia válida. La única relación rota son 2 registros de la matriz que no corresponden a ningún colaborador.

## 7. Fechas, valores numéricos y categorías
Compruebo fechas inválidas, fechas finales anteriores a las iniciales, sueldos negativos y valores de dominio. No cambio los datos en esta etapa.

In [7]:
def revisar_fecha(df, campo):
    s = df[campo]
    convertida = pd.to_datetime(s, errors="coerce")
    print(campo, "inválidas no vacías:", ((~faltante(s)) & convertida.isna()).sum())
    return convertida

for campo in ["fechanac", "fechainc", "fechabaja"]:
    revisar_fecha(c, campo)
for nombre, df in [("Proyectos", p), ("Asignaciones", a)]:
    print(nombre)
    inicio = revisar_fecha(df, "inicio")
    fin = revisar_fecha(df, "fin")
    print("Fin anterior al inicio:", (fin < inicio).sum())
for campo in ["sueldo_neto_total", "sueldo_total_integrado", "impuestos_imss"]:
    s = pd.to_numeric(c[campo], errors="coerce")
    print(campo, "negativos:", s.lt(0).sum(), "no numéricos:", (s.isna() & ~faltante(c[campo])).sum())
print("Costos de activos negativos:", pd.to_numeric(json_tablas["activos"].costo, errors="coerce").lt(0).sum())
print("Estatus de colaboradores:")
display(c.estatus.value_counts())
print("Niveles de habilidades:")
display(h.nivel.value_counts(dropna=False))

fechanac inválidas no vacías: 0
fechainc inválidas no vacías: 0
fechabaja inválidas no vacías: 0
Proyectos
inicio inválidas no vacías: 0
fin inválidas no vacías: 0
Fin anterior al inicio: 0
Asignaciones
inicio inválidas no vacías: 0
fin inválidas no vacías: 0
Fin anterior al inicio: 0
sueldo_neto_total negativos: 0 no numéricos: 0
sueldo_total_integrado negativos: 0 no numéricos: 0
impuestos_imss negativos: 0 no numéricos: 0
Costos de activos negativos: 0
Estatus de colaboradores:


,count
estatus,
Activo,75
Baja,32


Niveles de habilidades:


,count
nivel,
Básico,687
Intermedio,369
Experto,249
Avanzado,234


**Observación:** No hay fechas inválidas, fechas de fin anteriores al inicio ni montos negativos. Hay 75 colaboradores activos (70.09 %) y 32 bajas (29.91 %). En la matriz predomina el nivel Básico (44.64 %), seguido de Intermedio (23.98 %), Experto (16.18 %) y Avanzado (15.20 %), por lo que hay pocos perfiles con dominio alto.

## 8. Contraste entre Excel y JSON
Comparo los conteos, identificadores y los sueldos integrados entre ambas fuentes. El Excel de clientes contiene una fila por proyecto, por lo que se compara contra los proyectos separados del JSON. Después comparo las asignaciones y la matriz de habilidades del Excel contra las colecciones del JSON.

In [8]:
def obtener_excel(nombre):
    return next(df for fuente, df in excel.items() if fuente.startswith("Catalogo_" + nombre + ".xlsx"))

comparaciones = [("Colaboradores", c, "ID", "id"),
                 ("Clientes", p, "Cód Cliente", "codigo_cliente"),
                 ("Departamentos", json_tablas["departamentos"], "codigo", "codigo"),
                 ("Roles", json_tablas["roles"], "Código", "codigo"),
                 ("Habilidades", json_tablas["habilidades"], "codigo", "codigo"),
                 ("Activos", json_tablas["activos"], "Cód Activo", "codigo")]
resultado = []
for nombre, df_json, campo_excel, campo_json in comparaciones:
    df_excel = obtener_excel(nombre)
    ids_e = set(df_excel[campo_excel].astype(str))
    ids_j = set(df_json[campo_json].astype(str))
    resultado.append([nombre, len(df_excel), len(df_json), len(ids_e-ids_j), len(ids_j-ids_e)])
display(pd.DataFrame(resultado, columns=["Catálogo", "Filas Excel", "Filas JSON", "IDs solo Excel", "IDs solo JSON"]))

ce = obtener_excel("Colaboradores")
print("Bajas sin fecha en Excel:", (ce.Estatus.eq("Baja") & faltante(ce["Fecha de Baja"])).sum())
union = ce[["ID", "Sueldo Total Integrado"]].merge(c[["id", "sueldo_total_integrado"]], left_on="ID", right_on="id", validate="one_to_one")
diferencia = pd.to_numeric(union["Sueldo Total Integrado"], errors="coerce") - pd.to_numeric(union["sueldo_total_integrado"], errors="coerce")
print("Sueldos integrados distintos (tolerancia 0.01):", diferencia.abs().gt(0.01).sum())
print("Sueldos no comparables:", diferencia.isna().sum())

,Catálogo,Filas Excel,Filas JSON,IDs solo Excel,IDs solo JSON
0,Colaboradores,107,107,0,0
1,Clientes,12,12,0,0
2,Departamentos,6,6,0,0
3,Roles,55,55,0,0
4,Habilidades,71,71,0,0
5,Activos,123,123,0,0


Bajas sin fecha en Excel: 18
Sueldos integrados distintos (tolerancia 0.01): 0
Sueldos no comparables: 0


In [9]:
# Asignaciones: Excel contra JSON (se comparan por colaborador y proyecto)
ae = next(df for fuente, df in excel.items() if fuente.startswith("Asignaciones_Colaboradores.xlsx"))
id_por_uid = dict(zip(c["uid"], c["id"]))
a_cmp = a.assign(id=a["uid_colaborador"].map(id_por_uid),
                 proyecto=a["proyecto_val"].str.split("|").str[1].str.strip())
ae_cmp = ae.assign(proyecto=ae["Proyecto"].astype(str).str.strip())
union_a = a_cmp.merge(ae_cmp, left_on=["id", "proyecto"], right_on=["ID Colaborador", "proyecto"],
                      how="outer", indicator=True)
inicio_j, inicio_e = pd.to_datetime(union_a["inicio"], errors="coerce"), pd.to_datetime(union_a["Fecha Inicio"], errors="coerce")
fin_j, fin_e = pd.to_datetime(union_a["fin"], errors="coerce"), pd.to_datetime(union_a["Fecha Fin"], errors="coerce")
print("Asignaciones Excel:", len(ae), "| JSON:", len(a))
print("Asignaciones que solo están en una fuente:", union_a["_merge"].ne("both").sum())
print("Fecha de inicio distinta:", (inicio_j != inicio_e).sum())
print("Fecha de fin distinta (cuando ambas tienen fecha):", (fin_j.notna() & fin_e.notna() & (fin_j != fin_e)).sum())
print("Sin fecha fin en JSON:", faltante(a["fin"]).sum(), "| 'Indefinido' en Excel:", ae["Fecha Fin"].astype(str).eq("Indefinido").sum())
print("Asignaciones con fecha de inicio 1990-01-01:", ae["Fecha Inicio"].astype(str).str.startswith("1990-01-01").sum())
print("Asignaciones con rol PENDIENTE:", ae["Rol Asignado"].eq("PENDIENTE").sum())
asignados = set(a["uid_colaborador"])
print("Colaboradores activos sin asignación:", (activos & ~c["uid"].isin(asignados)).sum(), "de", activos.sum())
print("Colaboradores asignados sin matriz de habilidades:", len(asignados - set(matriz["uid_colaborador"])), "de", len(asignados))

# Matriz de habilidades: el Excel tiene una columna por habilidad; se pasa a una fila por combinación
me = next(df for fuente, df in excel.items() if fuente.startswith("Matriz_Habilidades.xlsx"))
codigos_hab = set(json_tablas["habilidades"]["codigo"])
cols_hab = [col for col in me.columns if str(col).split(" - ")[0] in codigos_hab]
me_largo = me.melt(id_vars="ID Colaborador", value_vars=cols_hab, var_name="columna", value_name="nivel")
me_largo = me_largo[~faltante(me_largo["nivel"])].copy()
me_largo["codigo_habilidad"] = me_largo["columna"].str.split(" - ").str[0]
h_cmp = h.assign(id=h["uid_colaborador"].map(id_por_uid))
union_h = h_cmp.dropna(subset=["id"]).merge(me_largo, left_on=["id", "codigo_habilidad"],
                                            right_on=["ID Colaborador", "codigo_habilidad"], how="outer", indicator=True)
ambos = union_h["_merge"].eq("both")
print("\nMatriz Excel:", me["ID Colaborador"].nunique(), "colaboradores,", len(cols_hab), "habilidades | JSON:", len(matriz), "registros")
print("Registros JSON sin colaborador (no están en el Excel):", h_cmp.loc[h_cmp["id"].isna(), "uid_colaborador"].nunique(),
      "con", h_cmp["id"].isna().sum(), "combinaciones")
print("Combinaciones Excel:", len(me_largo), "| JSON con colaborador:", h_cmp["id"].notna().sum())
print("Combinaciones que solo están en una fuente:", (~ambos).sum())
print("Niveles distintos:", (union_h.loc[ambos, "nivel_x"] != union_h.loc[ambos, "nivel_y"]).sum())
print("Colaboradores de la matriz por estatus (Excel):", me["Estatus"].value_counts().to_dict())

Asignaciones Excel: 31 | JSON: 31
Asignaciones que solo están en una fuente: 0
Fecha de inicio distinta: 0
Fecha de fin distinta (cuando ambas tienen fecha): 0
Sin fecha fin en JSON: 24 | 'Indefinido' en Excel: 24
Asignaciones con fecha de inicio 1990-01-01: 21
Asignaciones con rol PENDIENTE: 29
Colaboradores activos sin asignación: 45 de 75
Colaboradores asignados sin matriz de habilidades: 8 de 30

Matriz Excel: 54 colaboradores, 71 habilidades | JSON: 56 registros
Registros JSON sin colaborador (no están en el Excel): 2 con 10 combinaciones
Combinaciones Excel: 1529 | JSON con colaborador: 1529
Combinaciones que solo están en una fuente: 0
Niveles distintos: 0
Colaboradores de la matriz por estatus (Excel): {'Activo': 47, 'Baja': 7}


/tmp/ipykernel_4585/2199439146.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fin_j, fin_e = pd.to_datetime(union_a["fin"], errors="coerce"), pd.to_datetime(union_a["Fecha Fin"], errors="coerce")


**Observación:** Los catálogos, las asignaciones y la matriz coinciden entre Excel y JSON, pero eso solo indica que el respaldo es consistente con el Excel, no que los datos estén completos o correctos: los mismos huecos aparecen en ambas fuentes, como las 18 bajas sin fecha. Las 24 asignaciones sin fecha de fin en el JSON aparecen como "Indefinido" en Excel, lo que apoya que sean asignaciones indefinidas y no errores. Conviene revisar que 21 asignaciones (67.74 %) inician el 1990-01-01, una fecha que parece un valor por defecto, y que 29 de 31 (93.55 %) tienen el rol "PENDIENTE". 45 de los 75 colaboradores activos (60 %) no tienen asignación. En la matriz coinciden las 1,529 combinaciones y sus niveles; los 2 registros extra del JSON no existen en el Excel ni en colaboradores, así que podrían ser registros de prueba o eliminados.

## 9. Consulta SQL de comprobación (opcional)
Uso SQLite desde Python para comprobar los hallazgos sin guardar información personal en una base externa.

In [10]:
import sqlite3
with sqlite3.connect(":memory:") as con:
    c[["estatus", "depto", "puesto", "fechabaja"]].to_sql("colaboradores", con, index=False)
    display(pd.read_sql_query("""
        SELECT estatus, COUNT(*) AS colaboradores,
               SUM(CASE WHEN depto IS NULL OR TRIM(depto) = '' THEN 1 ELSE 0 END) AS sin_departamento,
               SUM(CASE WHEN puesto = 'PENDIENTE' THEN 1 ELSE 0 END) AS puesto_pendiente
        FROM colaboradores GROUP BY estatus
    """, con))

,estatus,colaboradores,sin_departamento,puesto_pendiente
0,Activo,75,64,29
1,Baja,32,31,19


**Observación:** La consulta confirma los hallazgos de la sección 5 y muestra que el problema afecta sobre todo al personal actual: 64 de los 75 colaboradores activos (85.33 %) no tienen departamento y 29 (38.67 %) tienen el puesto pendiente.

## 10. Conclusión individual
Los archivos de Excel y el respaldo JSON coinciden en conteos, identificadores, asignaciones, niveles de habilidad y sueldos integrados, y no se encontraron duplicados, fechas inválidas ni montos negativos. Aun así, esto no garantiza que la información sea completa. La mayoría de los colaboradores activos no tiene departamento, muchos puestos y roles asignados siguen como "PENDIENTE", más de la mitad de las bajas no tiene fecha de salida y el 37.33 % de los colaboradores activos no tiene habilidades registradas. También hay 2 registros de la matriz sin colaborador y 21 asignaciones con una fecha de inicio que parece por defecto.

Para analizar disponibilidad hay que confirmar con OT Consulting cómo se manejan las asignaciones indefinidas (77.42 %), que pueden ser válidas, y completar la matriz de habilidades. Para analizar rentabilidad faltan datos como horas de participación, costo por hora y montos de venta por proyecto. En la siguiente etapa se recomienda validar y limpiar estos campos antes de construir indicadores.